In [1]:
# importing necessary libraries 

import pandas as pd # to load data and analyse, manipulate and visualize the DataFrame
import numpy as np # to deal with arrays and numerical values
from lightgbm import LGBMRegressor  # lightgbm model for regression problem
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error  # to evaluate the model performance
from sklearn.model_selection import train_test_split  # to split a dataset into subdatasets
from sklearn.model_selection import cross_val_score #  to train and test a model over multiple folds of the dataset
import pickle # to save or open the trained model
import optuna # to optimize the hyperparameters

In [2]:
# to load the data into DataFrame with pandas package

data = pd.read_excel('07_ODU1_5min_for_IAT_prediction_model.xlsx')

In [3]:
# Provide a dimension of the DataFrame 
data.shape

(1540, 14)

In [4]:
# showing the column labels of the DataFrame
data.columns

Index(['ST', 'OAT', 'Delta_IAT_OAT', 'Delta_ST_IAT', 'Delta_ST_OAT',
       'High_Pressure', 'Day_sin', 'Day_cos', 'Op_Mode_0', 'Op_Mode_1',
       'Year_sin', 'Year_cos', 'OTW', 'IAT'],
      dtype='object')

In [5]:
# visualize the first 5 rows of DataFrame 
data.head()

,ST,OAT,Delta_IAT_OAT,Delta_ST_IAT,Delta_ST_OAT,High_Pressure,Day_sin,Day_cos,Op_Mode_0,Op_Mode_1,Year_sin,Year_cos,OTW,IAT
0,28.000000,-1.180000,23.691832,5.488168,29.180000,29.750000,0.806445,-0.591310,0,1,-0.179679,0.983725,-1.835570,22.511832
1,28.000000,-1.021429,27.105871,1.915557,29.021429,31.678571,0.793353,-0.608761,0,1,-0.179620,0.983736,-1.856887,26.084443
2,28.000000,-1.003448,29.688879,-0.685431,29.003448,30.306897,0.779884,-0.625923,0,1,-0.179562,0.983747,-1.863094,28.685431
3,28.000000,-0.956667,30.884415,-1.927749,28.956667,29.760000,0.766044,-0.642788,0,1,-0.179503,0.983757,-1.892156,29.927749
4,28.037447,-0.856667,31.464853,-2.570740,28.894113,29.923333,0.751840,-0.659346,0,1,-0.179444,0.983768,-1.864571,30.608186


In [6]:
# The normalizing the inputs to transform features to be on a similar scale of 0 to 1 range


def normalize_inputs(data):
    ST_min, ST_max = 18, 31
    IAT_min, IAT_max = 15, 35
    OAT_min, OAT_max = -20, 20
    OTW_min, OTW_max = -15, 10
    High_Pressure_min, High_Pressure_max = 5, 35
    
    
    data['ST'] = (data['ST'] - ST_min) / (ST_max - ST_min)
    data['IAT'] = (data['IAT'] - IAT_min) / (IAT_max - IAT_min)
    data['OAT'] = (data['OAT'] - OAT_min) / (OAT_max - OAT_min)
    data['OTW'] = (data['OTW'] - OTW_min) / (OTW_max - OTW_min)
    data['High_Pressure'] = (data['High_Pressure'] - High_Pressure_min) / (High_Pressure_max - High_Pressure_min)
    
    return data

# Denormalizing the predicted outputs for recovering the shape of the original data 

def inverse_normalization_output(outputs):
    IAT_min, IAT_max = 15, 35

    outputs['IAT'] = outputs['IAT'] * (IAT_max - IAT_min) + IAT_min

    return outputs

In [7]:
# Assigning the normalization function to variable data
data = normalize_inputs(data)

In [8]:
# function to transform time series data into a supervised learning dataset (input(X) and output(y))
# The function takes four arguments
# 1. Sequence of observations as a 2D NumPy array 
# 2. Number of lag observations as input (X) 
# 3.Number of observations as output (y) 
# 4. Boolean whether or not to drop rows with NaN values 


def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):      
	n_vars = 1 if type(data) is list else data.shape[1]
	df = pd.DataFrame(data)
	cols, names = list(), list()
	# input sequence (t-n, ... t-1)
	for i in range(n_in, 0, -1):       
		cols.append(df.shift(i))
		names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
	# forecast sequence (t, t+1, ... t+n)
	for i in range(0, n_out):
		cols.append(df.shift(-i))
		if i == 0:
			names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
		else:
			names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
	# put it all together
	agg = pd.concat(cols, axis=1)
	agg.columns = names
	# drop rows with NaN values
	if dropnan:
		agg.dropna(inplace=True)
	return agg         # The function returns a pandas DataFrame of series framed for supervised learning     

In [9]:
#  to convert the values of the series from int type to a float type with 32 bits in computer memory
values = data.values.astype('float32')
reframed = series_to_supervised(values,2, 1)   # to use two values at the previous time step
reframed.columns

Index(['var1(t-2)', 'var2(t-2)', 'var3(t-2)', 'var4(t-2)', 'var5(t-2)',
       'var6(t-2)', 'var7(t-2)', 'var8(t-2)', 'var9(t-2)', 'var10(t-2)',
       'var11(t-2)', 'var12(t-2)', 'var13(t-2)', 'var14(t-2)', 'var1(t-1)',
       'var2(t-1)', 'var3(t-1)', 'var4(t-1)', 'var5(t-1)', 'var6(t-1)',
       'var7(t-1)', 'var8(t-1)', 'var9(t-1)', 'var10(t-1)', 'var11(t-1)',
       'var12(t-1)', 'var13(t-1)', 'var14(t-1)', 'var1(t)', 'var2(t)',
       'var3(t)', 'var4(t)', 'var5(t)', 'var6(t)', 'var7(t)', 'var8(t)',
       'var9(t)', 'var10(t)', 'var11(t)', 'var12(t)', 'var13(t)', 'var14(t)'],
      dtype='object')

In [10]:
# remove 'Delta_IAT_OAT', 'Delta_ST_IAT' at the value of time t as we are predicting IAT

reframed.drop(['var3(t)', 'var4(t)'], axis =1, inplace=True) 
reframed = reframed.values     

In [11]:
# Distinguish the input features and output
features = reframed[:, :-1]
targets = reframed[:, -1]

In [12]:
# Split the data into 80 % of the train data and 20 % of the test data
train_x, test_x, train_y, test_y = train_test_split(features, targets, test_size=0.2, shuffle=True, random_state=123)

In [13]:
# to initilaize the starting time
import time
start = time.time()

In [14]:
# hyperparameters tuning
# to define objective function to be optimized

def objective(trial):

    param = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 3000, step=20),
        "n_estimators": trial.suggest_categorical('n_estimators', [10,30, 50, 100, 200, 500]),
        'learning_rate': trial.suggest_categorical('learning_rate', [0.001, 0.01, 0.1, 0.3, 0.008,0.009]),
        'max_depth': trial.suggest_categorical('max_depth', [3,5,7,9,11,13,15,17,20]),
        "reg_alpha": trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'reg_lambda':trial.suggest_loguniform('reg_lambda', 0.5, 1.0),
        
    }
    
    # Returns a list of 5 cross validate scores
    cv_score_lr = cross_val_score(LGBMRegressor(**param), train_x, train_y, cv=5,scoring='r2')
    
    # Average of the scores to provide a single value
    mean_r2 = round(sum(cv_score_lr)/len(cv_score_lr), 3)
    

    return mean_r2

In [15]:
# To optimize the defined objective

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=1000)
print('Number of finished trials:', len(study.trials))

[I 2023-02-01 22:09:24,411] A new study created in memory with name: no-name-825b197c-5f29-40b4-8160-c7db1738f819
[I 2023-02-01 22:09:25,433] Trial 0 finished with value: 0.281 and parameters: {'num_leaves': 1780, 'n_estimators': 30, 'learning_rate': 0.009, 'max_depth': 20, 'reg_alpha': 0.005211028100668869, 'reg_lambda': 0.5485982860657752}. Best is trial 0 with value: 0.281.
[I 2023-02-01 22:09:26,149] Trial 1 finished with value: 0.659 and parameters: {'num_leaves': 2240, 'n_estimators': 10, 'learning_rate': 0.3, 'max_depth': 13, 'reg_alpha': 2.5235508405613527, 'reg_lambda': 0.9562924741460485}. Best is trial 1 with value: 0.659.
[I 2023-02-01 22:09:26,609] Trial 2 finished with value: 0.412 and parameters: {'num_leaves': 460, 'n_estimators': 50, 'learning_rate': 0.01, 'max_depth': 17, 'reg_alpha': 0.2573132249662529, 'reg_lambda': 0.8713932269138257}. Best is trial 1 with value: 0.659.
[I 2023-02-01 22:09:27,315] Trial 3 finished with value: 0.844 and parameters: {'num_leaves': 22

[I 2023-02-01 22:10:35,546] Trial 30 finished with value: 0.013 and parameters: {'num_leaves': 3000, 'n_estimators': 10, 'learning_rate': 0.001, 'max_depth': 5, 'reg_alpha': 0.007261687123520701, 'reg_lambda': 0.5344858603165434}. Best is trial 23 with value: 0.873.
[I 2023-02-01 22:10:37,905] Trial 31 finished with value: 0.874 and parameters: {'num_leaves': 2400, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.05932479345056157, 'reg_lambda': 0.5488362472757008}. Best is trial 31 with value: 0.874.
[I 2023-02-01 22:10:40,339] Trial 32 finished with value: 0.869 and parameters: {'num_leaves': 2720, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.04428392347849782, 'reg_lambda': 0.5688939606508514}. Best is trial 31 with value: 0.874.
[I 2023-02-01 22:10:42,463] Trial 33 finished with value: 0.779 and parameters: {'num_leaves': 2380, 'n_estimators': 500, 'learning_rate': 0.01, 'max_depth': 5, 'reg_alpha': 0.13146821456876556, 'reg_lam

[I 2023-02-01 22:11:19,532] Trial 60 finished with value: 0.741 and parameters: {'num_leaves': 1060, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 15, 'reg_alpha': 1.6321265642092362, 'reg_lambda': 0.7583098561392886}. Best is trial 54 with value: 0.875.
[I 2023-02-01 22:11:20,779] Trial 61 finished with value: 0.872 and parameters: {'num_leaves': 660, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.001629639335706152, 'reg_lambda': 0.7045686753739634}. Best is trial 54 with value: 0.875.
[I 2023-02-01 22:11:21,999] Trial 62 finished with value: 0.871 and parameters: {'num_leaves': 320, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.004517139242175539, 'reg_lambda': 0.7169734677733212}. Best is trial 54 with value: 0.875.
[I 2023-02-01 22:11:23,346] Trial 63 finished with value: 0.872 and parameters: {'num_leaves': 2480, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.009466734889135693, 'reg_lambd

[I 2023-02-01 22:12:25,777] Trial 90 finished with value: 0.856 and parameters: {'num_leaves': 1120, 'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 11, 'reg_alpha': 0.0954008331063407, 'reg_lambda': 0.5031745168454627}. Best is trial 54 with value: 0.875.
[I 2023-02-01 22:12:27,157] Trial 91 finished with value: 0.873 and parameters: {'num_leaves': 140, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.0014894042544655586, 'reg_lambda': 0.524833451795935}. Best is trial 54 with value: 0.875.
[I 2023-02-01 22:12:28,388] Trial 92 finished with value: 0.871 and parameters: {'num_leaves': 140, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.00304284631396834, 'reg_lambda': 0.5264215968640796}. Best is trial 54 with value: 0.875.
[I 2023-02-01 22:12:29,643] Trial 93 finished with value: 0.873 and parameters: {'num_leaves': 2740, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.0064777986484649865, 'reg_lambd

[I 2023-02-01 22:13:29,587] Trial 120 finished with value: 0.877 and parameters: {'num_leaves': 3000, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.004949201417809822, 'reg_lambda': 0.5161101397677877}. Best is trial 120 with value: 0.877.
[I 2023-02-01 22:13:30,929] Trial 121 finished with value: 0.876 and parameters: {'num_leaves': 3000, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.004903948779521753, 'reg_lambda': 0.5206569918133339}. Best is trial 120 with value: 0.877.
[I 2023-02-01 22:13:32,002] Trial 122 finished with value: 0.876 and parameters: {'num_leaves': 3000, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.08261946504360602, 'reg_lambda': 0.5118184798028613}. Best is trial 120 with value: 0.877.
[I 2023-02-01 22:13:33,391] Trial 123 finished with value: 0.876 and parameters: {'num_leaves': 3000, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.0049583642613677415, 

...

[I 2023-02-01 22:46:09,800] Trial 900 finished with value: 0.853 and parameters: {'num_leaves': 1180, 'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.01035766779287971, 'reg_lambda': 0.542175031321767}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:46:21,293] Trial 901 finished with value: 0.851 and parameters: {'num_leaves': 2940, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 13, 'reg_alpha': 0.003289179122098414, 'reg_lambda': 0.523290608596005}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:46:21,516] Trial 902 finished with value: 0.819 and parameters: {'num_leaves': 2760, 'n_estimators': 30, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.006575593557987846, 'reg_lambda': 0.530243797349994}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:46:22,463] Trial 903 finished with value: 0.874 and parameters: {'num_leaves': 1240, 'n_estimators': 200, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.08139152672430039, 'reg_lam

[I 2023-02-01 22:47:38,223] Trial 930 finished with value: 0.877 and parameters: {'num_leaves': 2580, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.009845327976948691, 'reg_lambda': 0.5555817864980117}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:47:40,457] Trial 931 finished with value: 0.876 and parameters: {'num_leaves': 2880, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.008383697633207616, 'reg_lambda': 0.5126962629902345}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:47:42,941] Trial 932 finished with value: 0.877 and parameters: {'num_leaves': 2780, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.007116840940227168, 'reg_lambda': 0.5479727353570706}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:47:49,832] Trial 933 finished with value: 0.222 and parameters: {'num_leaves': 2740, 'n_estimators': 200, 'learning_rate': 0.001, 'max_depth': 13, 'reg_alpha': 0.007730133946040372, 

[I 2023-02-01 22:48:47,618] Trial 960 finished with value: 0.876 and parameters: {'num_leaves': 2880, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.006327190016809869, 'reg_lambda': 0.5204545215777555}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:48:53,795] Trial 961 finished with value: 0.222 and parameters: {'num_leaves': 2020, 'n_estimators': 200, 'learning_rate': 0.001, 'max_depth': 17, 'reg_alpha': 0.01001729286476477, 'reg_lambda': 0.5483786356265697}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:48:56,268] Trial 962 finished with value: 0.876 and parameters: {'num_leaves': 2720, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.016169016381968493, 'reg_lambda': 0.5752235398801102}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:48:57,286] Trial 963 finished with value: 0.874 and parameters: {'num_leaves': 2720, 'n_estimators': 200, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.012696133981496816, '

[I 2023-02-01 22:50:15,765] Trial 990 finished with value: 0.88 and parameters: {'num_leaves': 1660, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.00976728997055893, 'reg_lambda': 0.5469706007010869}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:50:18,235] Trial 991 finished with value: 0.876 and parameters: {'num_leaves': 1660, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.008273316616589407, 'reg_lambda': 0.5579732756262022}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:50:19,129] Trial 992 finished with value: 0.749 and parameters: {'num_leaves': 2540, 'n_estimators': 10, 'learning_rate': 0.3, 'max_depth': 17, 'reg_alpha': 0.0016482825066389804, 'reg_lambda': 0.528649902916133}. Best is trial 255 with value: 0.88.
[I 2023-02-01 22:50:21,354] Trial 993 finished with value: 0.873 and parameters: {'num_leaves': 1640, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.012667286809816639, 'reg_

Number of finished trials: 1000


In [16]:
# to print out the time taken for optimizing

end = time.time()
print(end - start)

2470.925170660019


In [17]:
# print out the best hyperparameters
print('Best trial:', study.best_trial.params)

Best trial: {'num_leaves': 1920, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.009808063939796512, 'reg_lambda': 0.5538088703208787}


In [18]:
# Fit the model (LGBMRegressor) with best hyperparameters and evaluate the performance
param = {
    "num_leaves": 1920,
    "n_estimators": 500,
    'learning_rate': 0.3,
    'max_depth': 3,
    "reg_alpha": 0.009808063939796512,
    'reg_lambda':0.5538088703208787,
 
}
model = LGBMRegressor(**param)

model.fit(train_x,train_y,eval_set=[(test_x,test_y)], verbose=-1)

C:\Users\HuBEL\anaconda3\lib\site-packages\lightgbm\sklearn.py:736: UserWarning: 'verbose' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose' argument is deprecated and will be removed in a future release of LightGBM. "


LGBMRegressor(learning_rate=0.3, max_depth=3, n_estimators=500, num_leaves=1920,
              reg_alpha=0.009808063939796512, reg_lambda=0.5538088703208787)

In [19]:
# Predicting on train dataset

yhat_train = model.predict(train_x)

predicted_train=pd.DataFrame(yhat_train,columns=['IAT'])
predicted_train = inverse_normalization_output(predicted_train)

actual_train = pd.DataFrame(train_y,columns=['IAT'])
actual_train = inverse_normalization_output(actual_train)

print('Training results')
print('r^2:',r2_score(actual_train, predicted_train))
print('RMSE:',np.sqrt(mean_squared_error(actual_train, predicted_train)))
print('MAE:',mean_absolute_error(actual_train, predicted_train))


Training results
r^2: 0.9974149613375751
RMSE: 0.10465326494716622
MAE: 0.07576711557757082


In [20]:
# Predicting on test dataset

yhat_test = model.predict(test_x)
predicted_test=pd.DataFrame(yhat_test,columns=['IAT'])
predicted_test = inverse_normalization_output(predicted_test)

actual_test=pd.DataFrame(test_y,columns=['IAT'])
actual_test = inverse_normalization_output(actual_test)

print('Testing results')
print('r^2:',r2_score(actual_test, predicted_test))
print('RMSE:',np.sqrt(mean_squared_error(actual_test, predicted_test)))
print('MAE:',mean_absolute_error(actual_test, predicted_test))

Testing results
r^2: 0.8586161496644713
RMSE: 0.6412464614015015
MAE: 0.34094599129102277


In [21]:
# to save the model for use at a later time or place
pickle.dump(model, open('IAT.pkl','wb'))

In [22]:
# To concate, save the actual and predicted values into excel file

train_df = pd.concat([actual_train, predicted_train], axis=1)
train_df.to_excel("training_results_past_2.xlsx",index=False)
test_df = pd.concat([actual_test, predicted_test], axis=1)
test_df.to_excel("testing_results_past_2.xlsx",index=False)